# Luckyscent webscrapen
Scrapes naam, merk, notes, stijl, beschrijving, prijs, concentratie, gender, land.

In [1]:
# Installeer benodigde packages
!pip install requests beautifulsoup4 pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Laptop\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
!pip install beautifulsoup4


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\Laptop\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import subprocess
subprocess.run([r"C:\Users\Laptop\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe", 
                "-m", "pip", "install", "beautifulsoup4"], capture_output=True)

CompletedProcess(args=['C:\\Users\\Laptop\\AppData\\Local\\Microsoft\\WindowsApps\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\python.exe', '-m', 'pip', 'install', 'beautifulsoup4'], returncode=0, stdout=b'Defaulting to user installation because normal site-packages is not writeable\r\nRequirement already satisfied: beautifulsoup4 in c:\\users\\laptop\\appdata\\local\\packages\\pythonsoftwarefoundation.python.3.13_qbz5n2kfra8p0\\localcache\\local-packages\\python313\\site-packages (4.14.3)\r\nRequirement already satisfied: soupsieve>=1.6.1 in c:\\users\\laptop\\appdata\\local\\packages\\pythonsoftwarefoundation.python.3.13_qbz5n2kfra8p0\\localcache\\local-packages\\python313\\site-packages (from beautifulsoup4) (2.8.3)\r\nRequirement already satisfied: typing-extensions>=4.0.0 in c:\\users\\laptop\\appdata\\local\\packages\\pythonsoftwarefoundation.python.3.13_qbz5n2kfra8p0\\localcache\\local-packages\\python313\\site-packages (from beautifulsoup4) (4.12.2)\r\n', stderr=b'\r\n[

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}
BASE_URL = "https://www.luckyscent.com"
DELAY = 1.5  # seconden tussen requests

print('Imports OK')

Imports OK


## Verzamel alle product-URLs

In [5]:
def get_product_urls(max_pages=None):
    """Pagineert door /fragrances en verzamelt alle product-URLs."""
    urls = []
    cursor_url = f"{BASE_URL}/fragrances"
    page = 0

    while cursor_url:
        page += 1
        if max_pages and page > max_pages:
            break

        print(f"Listing pagina {page}: {cursor_url}")
        resp = requests.get(cursor_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(resp.text, "html.parser")

        for a in soup.select("a[href^='/products/']"):
            full = BASE_URL + a["href"]
            if full not in urls:
                urls.append(full)

        next_link = soup.select_one("a[href*='cursor=']") 
        cursor_url = BASE_URL + next_link["href"] if next_link else None

        time.sleep(DELAY)

    print(f"\nTotaal gevonden: {len(urls)} product-URLs")
    return urls

In [6]:
# MAX_PAGES = 3  # voor een snelle test
MAX_PAGES = None  # alle pagina's

product_urls = get_product_urls(max_pages=MAX_PAGES)
print(product_urls[:5])  # preview

Listing pagina 1: https://www.luckyscent.com/fragrances
Listing pagina 2: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDIwMzUxNDA0NDczNywibGFzdF92YWx1ZSI6IjIwMjYtMDMtMjMgMjE6MTU6MzkuMDAwMDAwIiwib2Zmc2V0IjozNX0%3D
Listing pagina 3: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDE4NTMxNjUzMjU0NSwibGFzdF92YWx1ZSI6IjIwMjYtMDMtMDcgMDA6MjU6NDEuMDAwMDAwIiwib2Zmc2V0Ijo3MX0%3D
Listing pagina 4: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDE3NzgwMTE1ODk3NywibGFzdF92YWx1ZSI6IjIwMjYtMDItMjMgMjE6MTU6NDcuMDAwMDAwIiwib2Zmc2V0IjoxMDd9
Listing pagina 5: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDE3MTQ2NDYxNDIwOSwibGFzdF92YWx1ZSI6IjIwMjYtMDItMTQgMDA6MDE6MDQuMDAwMDAwIiwib2Zmc2V0IjoxNDN9
Listing pagina 6: https://www.luckyscent.com/fragrances?direction=next&cursor=eyJsYXN0X2lkIjoxMDEzMDM1NjE0MjQwMSwibGFzdF92YWx1ZSI6IjIwMjYtMDEtMjEgMTY6MTA6NTguMDAwMDAwIiwib2Zmc2V0IjoxNzl9
Listi

## Scrape elke productpagina

In [23]:
def get_price(soup):
    import re
    for tag in soup.find_all(string=True):
        t = tag.strip()
        m = re.match(r'^\$(\d+)$', t)
        if m:
            return int(m.group(1))
    return None

def scrape_product(url):
    slug = url.rstrip("/").split("/products/")[-1]
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(resp.text, "html.parser")

        name = soup.select_one("h1").get_text(strip=True) if soup.select_one("h1") else ""

        # Brand uit URL halen
        import re
        brand = ""
        m = re.search(r"-by-(.+)$", slug)
        if m:
            brand = m.group(1).replace("-", " ").title()

        price = get_price(soup)

        concentration = gender = country = released = ""
        for a in soup.select("a[href*='fragrances?f.l.']"):
            href, txt = a["href"], a.get_text(strip=True)
            if "concentration=" in href: concentration = txt
            elif "gender=" in href: gender = txt
            elif "country=" in href: country = txt
            elif "year_released=" in href: released = txt

        notes = list(dict.fromkeys(a.get_text(strip=True) for a in soup.select("a[href*='notes=']")))
        styles = list(dict.fromkeys(a.get_text(strip=True) for a in soup.select("a[href*='style=']")))

        description = ""
        scoop = soup.find(lambda t: t.name in ["h2","h3","h4"] and "Scoop" in t.get_text())
        if scoop:
            nxt = scoop.find_next_sibling()
            if nxt:
                description = nxt.get_text(strip=True)
            else:
                description = scoop.find_next(string=True, recursive=True)
            if description:
                description = description.strip()
             

        return {"name": name, "brand": brand, "concentration": concentration,
                "gender": gender, "country": country, "released": released,
                "price": price, "notes": ", ".join(notes), "style": ", ".join(styles),
                "description": description, "url": url}
    except Exception as e:
        print(f"Fout: {url} — {e}")
        return None

In [24]:
# Snelle test op 1 product
test = scrape_product("https://www.luckyscent.com/products/kun-amo-by-jeroboam")
for k, v in test.items():
    print(f"{k}: {v}")

name: Kun Amo
brand: Jeroboam
concentration: Extrait
gender: Unisex
country: France
released: 2025
price: 130
notes: Mandarin, Bergamot, Almond, Crunchy Pear, Jasmine, Sea Spray, Raspberry, Cedarwood, Sugar, White Musks, Ambery Woods, Cashmeran
style: Floral - Fruity, Fruity, Powdery / Soft, Sweet, Woody - Amber
description: An homage to Jeroboam founder FranÃ§ois HÃ©nin's childhood memories of picking crisp pears from his family orchard, Kun Amo is a luminous and sensual perfume extract in which pear reigns supreme, transformed into a veritable olfactory muse. Sun-drenched, luscious and seductive, the fruit mingles with a generous dose of sweetness and captivating aromas, underpinned by modern, ambery woods and the houseâs emblematic musks, boasting an intense trail and remarkable hold to leave a lasting impression. Radiating with confidence and joy, Kun Amo is a bold perfume that asserts its presence with elegance and character.
url: https://www.luckyscent.com/products/kun-amo-by-j

## Alles scrapen en opslaan

In [25]:
results = []

for i, url in enumerate(product_urls):
    print(f"[{i+1}/{len(product_urls)}] {url}")
    data = scrape_product(url)
    if data:
        results.append(data)
    time.sleep(DELAY)

    # Checkpoint elke 100 producten
    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv("luckyscent_checkpoint.csv", index=False)
        print(f"  --> Checkpoint opgeslagen ({len(results)} producten)")

print(f"\nKlaar! {len(results)} producten gescraped.")

[1/3355] https://www.luckyscent.com/products/fragrance-fitting
[2/3355] https://www.luckyscent.com/products/kun-amo-by-jeroboam
[3/3355] https://www.luckyscent.com/products/mango-sticky-rice-by-dannam
[4/3355] https://www.luckyscent.com/products/vanilified-by-kerosene
[5/3355] https://www.luckyscent.com/products/with-angels-and-archangels-by-kerosene
[6/3355] https://www.luckyscent.com/products/inner-child-by-lepoque-parfums
[7/3355] https://www.luckyscent.com/products/bianco-latte-by-giardini-di-toscana
[8/3355] https://www.luckyscent.com/products/white-rice-by-dannam
[9/3355] https://www.luckyscent.com/products/celestial-object-by-liis
[10/3355] https://www.luckyscent.com/products/hay-zhasmina-by-chopova-lowena
[11/3355] https://www.luckyscent.com/products/great-root-green-ruth-by-chopova-lowena
[12/3355] https://www.luckyscent.com/products/queen-rosa-rosa-is-queen-by-chopova-lowena
[13/3355] https://www.luckyscent.com/products/discovery-set-by-chopova-lowena
[14/3355] https://www.lu

## Opslaan als CSV

In [26]:
df = pd.DataFrame(results)
df.to_csv("luckyscent_fragrances.csv", index=False)
print(f"Opgeslagen: luckyscent_fragrances.csv ({len(df)} rijen)")
df.head()

Opgeslagen: luckyscent_fragrances.csv (3354 rijen)


,name,brand,concentration,gender,country,released,price,notes,style,description,url
0,Fragrance Fitting - Custom Sample Pack,,,,,,60,,,Let us help you find your next signature scent...,https://www.luckyscent.com/products/fragrance-...
1,Kun Amo,Jeroboam,Extrait,Unisex,France,2025,130,"Mandarin, Bergamot, Almond, Crunchy Pear, Jasm...","Floral - Fruity, Fruity, Powdery / Soft, Sweet...",An homage to Jeroboam founder FranÃ§ois HÃ©nin...,https://www.luckyscent.com/products/kun-amo-by...
2,Mango Sticky Rice,Dannam,Eau de Parfum,Unisex,,2026,160,"Ripe Mango, Sticky Rice, Coconut Milk","Creamy, Fruity, Gourmand, Milky, Sweet, Tropic...",Mango Sticky Rice pays homage to Thailandâs ...,https://www.luckyscent.com/products/mango-stic...
3,Vanilified,Kerosene,Eau de Parfum,Unisex,,2026,164,"Vanilla, Vanilla Beans, Vanilla Bourbon, Vanil...","Boozy, Creamy, Gourmand, Milky, Sweet",You enter an ice cream shop. You order a tripl...,https://www.luckyscent.com/products/vanilified...
4,With Angels and Archangels,Kerosene,Eau de Parfum,Unisex,,2026,164,"Sandalwood, Cedar, Incense, Charcoal, Rose, Musk","Floral - Spicy, Smoky, Spicy, Woody - Fresh, W...",It's you and the Divine together sharing a hol...,https://www.luckyscent.com/products/with-angel...


In [19]:
# Snelle statistieken
print(f"Totaal producten: {len(df)}")
print(f"Unieke merken: {df['brand'].nunique()}")
print(f"Missende notes: {df['notes'].eq('').sum()}")
print(f"Missende beschrijving: {df['description'].eq('').sum()}")
print("\nTop 10 merken:")
print(df['brand'].value_counts().head(10))

Totaal producten: 3353
Unieke merken: 2
Missende notes: 350
Missende beschrijving: 6

Top 10 merken:
brand
Brands    3352
             1
Name: count, dtype: int64
